# SSL Universal Training Notebook — Jupyter 서버 (RTX A5000 24GB) + VSCode

**사용법**: Cell 2의 `BRANCH` 변수와, Cell 3의 `CONFIG` 변수만 바꿔서 같은 노트북으로 모든 method 학습.

- `configs/mocov3_vits8.yaml`  — **★ MoCo v3 + ViT-S/8 (최종 제출 — STL10 89.74 / CIFAR10 87.77)**
- `configs/mocov3_vits.yaml`   — ViT-S/16 (이전 메인)
- `configs/mocov2_mc_r50.yaml` — Track A (multi-crop MoCo, 비교)
- `configs/vicreg_r50.yaml`    — Track B (VICReg)
- `configs/mocov2_r50.yaml`    — 기존 baseline 재현용

**디렉토리**: `data/ outputs/ logs/ features/` 는 **레포 루트에 자동 생성**되어 서버 디스크에
영구 보존된다 (Colab의 Drive 마운트·심링크 단계 불필요).

**장시간 학습 권장 방식**: 본학습(500 epoch)은 노트북 셀 대신 **JupyterLab Terminal + nohup**:
```bash
nohup python -u scripts/pretrain.py --config configs/mocov3_vits8.yaml > logs/pretrain.out 2>&1 &
tail -f logs/pretrain.out
```
노트북 셀(Cell 3)은 기본 풀학습(SANITY_EPOCHS=None). 메모리/시간 확인만 원하면 3으로. 중단 시 재실행하면 자동 resume.

**워크플로**: VSCode 편집 → `git push` to GitHub 브랜치 → 서버에서 Cell 2 재실행으로 `git pull`.
(GitHub가 single source of truth)

In [ ]:
# Cell 1 — GPU 확인
import torch, subprocess
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if torch.cuda.is_available():
    print('VRAM:', torch.cuda.get_device_properties(0).total_memory/1e9, 'GB')
    print('BF16 OK:', torch.cuda.is_bf16_supported())
print(subprocess.check_output(['nvidia-smi', '-L']).decode())
# 기대값: NVIDIA RTX A5000 / 24GB / BF16 OK: True (Ampere)

In [ ]:
# Cell 2 — 코드 동기화(git pull) + editable install + 작업 디렉토리 생성
#
# ★ 서버에 레포가 이미 clone 되어 있다는 전제 (이 노트북 자체가 레포 안에 있음).
# ★ VSCode에서 push한 최신 코드를 받아오려면 이 셀만 재실행.

BRANCH = 'final-jupyter'

import os, subprocess
from pathlib import Path

# 레포 루트로 이동 (notebooks/ 의 한 단계 위)
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)

# 최신 코드 동기화
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)

# 어느 commit에서 학습하는지 기록 — RESULTS.md에 옮겨 적기
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
print(f'Branch: {BRANCH}')
print(f'Commit: {commit}')

# Editable install (venv 활성 상태 가정 — 커널 "Python (ssl)")
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', '.'], check=True)

# 산출물 디렉토리 — 레포 루트에 생성 (서버 디스크 영구 보존, 심링크 불필요)
for sub in ['data', 'outputs', 'logs', 'features']:
    os.makedirs(sub, exist_ok=True)

print('Setup complete. Working dir:', os.getcwd())

In [ ]:
# Cell 3 — 학습 시작
#
# ★ 아래 CONFIG 한 줄만 바꾸면 OUT_KEY / 저장경로 / (Cell 4)추출경로가 전부 자동 연동됩니다.
import glob, subprocess, os, yaml

CONFIG = 'configs/mocov3_vits8.yaml'    # ⭐ ViT-S/8 (patch8, fine-grained 실험)
# CONFIG = 'configs/mocov3_vits.yaml'   # ViT-S/16 (이전 메인, STL10/CIFAR10 86.6%)
# CONFIG = 'configs/mocov2_mc_r34.yaml' # R34 multi-crop
# CONFIG = 'configs/vicreg_r50.yaml'    # VICReg

# OUT_KEY를 config의 output.dir에서 자동 유도 (경로 불일치/덮어쓰기 방지)
OUT_KEY = yaml.safe_load(open(CONFIG))['output']['dir'].replace('./', '')
print(f'CONFIG  = {CONFIG}')
print(f'OUT_KEY = {OUT_KEY}   (저장: 레포 루트 {OUT_KEY})')

SANITY_EPOCHS = None   # None = 풀학습(config epochs 전체). 메모리/시간만 확인하려면 3

# 자동 resume — outputs/에 마지막 ckpt 있으면 이어서 시작
ckpts = sorted(glob.glob(f'{OUT_KEY}/ckpt_ep*.pth'),
               key=lambda p: int(p.rsplit('_ep', 1)[1].split('.')[0]))
resume_args = ['--resume', ckpts[-1]] if ckpts else []
print('Resume:', ckpts[-1] if ckpts else 'None (fresh start)')

extra = ['--epochs', str(SANITY_EPOCHS)] if SANITY_EPOCHS is not None else []
if SANITY_EPOCHS is not None:
    print(f'>>> SANITY MODE — {SANITY_EPOCHS} epochs only')

cmd = ['python', '-u', 'scripts/pretrain.py', '--config', CONFIG] + resume_args + extra
print('CMD:', ' '.join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        bufsize=1, text=True)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('Exit code:', proc.returncode)

In [ ]:
# Cell 4 — Feature 추출 (Cell 3의 CONFIG/OUT_KEY 자동 연동)
import subprocess, os, glob, yaml

# Cell 3에서 정한 CONFIG/OUT_KEY 재사용 (Cell 4만 단독 실행 시 여기서 지정)
try:
    CONFIG, OUT_KEY
except NameError:
    CONFIG = 'configs/mocov3_vits8.yaml'
    OUT_KEY = yaml.safe_load(open(CONFIG))['output']['dir'].replace('./', '')

# ★ feature 추출 방식 (Phase A LP 레버, ViT 전용). 채택값 = last4_cls_patchmean.
#   cls(384, baseline) / avg(384) / cls_patchmean(768) / last4_cls(1536, DINO표준) /
#   last4_cls_patchmean(1920, ★채택). ResNet/Swin이면 무시되고 cls로 추출됨.
#   5종 sweep을 원하면 MODES=['cls',...] 로 두고 이 셀 본문을 for문으로 감싸면 됨.
FEATURE_MODE = 'last4_cls_patchmean'

# ★ 추출 해상도 레버. 96 = 학습 해상도(기본), 128 = 업스케일 평가.
#   128이면 STL10 96→128 / CIFAR 32→128 로 Resize 후 추출.
#   ViT는 96이 아닐 때 pos_embed를 자동 보간(--dynamic-img-size, 아래에서 자동 부착). 재학습 불필요.
EVAL_RES = 128   # ← 96으로 되돌리려면 이 숫자만 변경

OUT_DIR = OUT_KEY
TAG = OUT_KEY.split('/')[-1].replace('_seed42', '')   # 예: mocov3_vits8
ckpts = sorted(glob.glob(f'{OUT_DIR}/backbone_ep*.pth'),
               key=lambda p: int(p.rsplit('_ep', 1)[1].split('.')[0]))
assert ckpts, f'backbone_ep*.pth 없음: {OUT_DIR} (학습 먼저)'
BACKBONE = ckpts[-1]
EP = BACKBONE.rsplit('_ep', 1)[1].split('.')[0]
# mode+해상도를 경로에 포함 → 96/128 feature가 서로 안 덮어씀 (Cell 5/6이 이 FEAT_DIR 재사용)
FEAT_DIR = f'features/{TAG}_ep{EP}_{FEATURE_MODE}_r{EVAL_RES}'
print(f'backbone:     {BACKBONE}  (epoch {EP})')
print(f'feature_mode: {FEATURE_MODE}')
print(f'eval_res:     {EVAL_RES}px')
print(f'FEAT_DIR:     {FEAT_DIR}')

NORMALIZE = 'standardize'   # {'none','l2','standardize'}
os.makedirs(FEAT_DIR, exist_ok=True)
cmd = ['python', '-u', 'scripts/extract_features.py',
       '--backbone', BACKBONE, '--config', CONFIG,
       '--output-dir', FEAT_DIR, '--normalize', NORMALIZE,
       '--feature-mode', FEATURE_MODE, '--batch-size', '256',
       '--image-size', str(EVAL_RES)]
if EVAL_RES != 96:
    cmd += ['--dynamic-img-size']   # ViT pos_embed를 EVAL_RES에 맞게 보간 (96이면 불필요)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        bufsize=1, text=True)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'\nexit {proc.returncode}')
print('Feature dir:', FEAT_DIR)

In [ ]:
# Cell 5 — LP 평가 (evaluate.py 고정 recipe — 수정 절대 금지)
import subprocess, glob, os

# Cell 4의 FEAT_DIR 재사용 (단독 실행 시 가장 최근 추출 폴더 자동 선택)
try:
    FEAT_DIR
except NameError:
    feat_dirs = sorted(glob.glob('features/*_ep*'),
                       key=os.path.getmtime)
    assert feat_dirs, 'features 폴더 없음 (Cell 4 먼저 실행)'
    FEAT_DIR = feat_dirs[-1]
print(f'Evaluating:   {FEAT_DIR}')
print(f'feature_mode: {globals().get("FEATURE_MODE", "(FEAT_DIR에서 추론)")}')
# evaluate.py LP recipe는 불변. feature 차원은 npy에서 자동 감지되므로
# last4_cls_patchmean(1920-d)도 추가 수정 없이 그대로 평가됨.

cmd = ['python', '-u', 'evaluate.py',
    '--stl10-train-features',   f'{FEAT_DIR}/stl10_train_features.npy',
    '--stl10-train-labels',     f'{FEAT_DIR}/stl10_train_labels.npy',
    '--stl10-test-features',    f'{FEAT_DIR}/stl10_test_features.npy',
    '--stl10-test-labels',      f'{FEAT_DIR}/stl10_test_labels.npy',
    '--cifar10-train-features', f'{FEAT_DIR}/cifar10_train_features.npy',
    '--cifar10-train-labels',   f'{FEAT_DIR}/cifar10_train_labels.npy',
    '--cifar10-test-features',  f'{FEAT_DIR}/cifar10_test_features.npy',
    '--cifar10-test-labels',    f'{FEAT_DIR}/cifar10_test_labels.npy',
    '--device', 'cuda']
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        bufsize=1, text=True)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'\nexit {proc.returncode}')

In [ ]:
# Cell 6 — feature 시각화 (Cell 4의 FEAT_DIR 자동 연동)
#
# 경로 하드코딩 없음: Cell 4에서 추출한 FEAT_DIR을 그대로 사용.
# Cell 6만 단독 실행하면 가장 최근 추출 폴더(features/<tag>_ep<N>_<mode>)를 자동 선택.
# 차원 무관(_train_lp가 shape[1]로 D 자동) → last4_cls_patchmean(1920-d)도 그대로 동작.
import os, glob, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

try:
    FEAT_DIR
except NameError:
    feat_dirs = sorted(glob.glob('features/*_ep*'),
                       key=os.path.getmtime)
    assert feat_dirs, 'features 폴더 없음 (Cell 4 먼저 실행)'
    FEAT_DIR = feat_dirs[-1]
DATA = 'data'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODE = globals().get('FEATURE_MODE', os.path.basename(FEAT_DIR).split('_ep')[-1])
print(f'Visualizing:  {FEAT_DIR}   (device {device})')
print(f'feature_mode: {MODE}')

STL10_CLASSES   = ['airplane','bird','car','cat','deer','dog','horse','monkey','ship','truck']
CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

def _load(ds, split):
    x = np.load(f'{FEAT_DIR}/{ds}_{split}_features.npy').astype('float32')
    y = np.load(f'{FEAT_DIR}/{ds}_{split}_labels.npy').reshape(-1).astype('int64')
    return x, y

def _train_lp(tr_x, tr_y, te_x, epochs=100, bs=128, seed=42):
    # evaluate.py 동일 recipe: SGD lr0.1 mom0.9 wd0, cosine 100ep, bs128
    torch.manual_seed(seed)
    D, C = tr_x.shape[1], int(tr_y.max())+1
    head = nn.Linear(D, C).to(device)
    opt = torch.optim.SGD(head.parameters(), lr=0.1, momentum=0.9, weight_decay=0.0)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    Xtr = torch.tensor(tr_x, device=device); Ytr = torch.tensor(tr_y, device=device)
    n = Xtr.shape[0]
    for ep in range(epochs):
        head.train(); perm = torch.randperm(n, device=device)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            loss = F.cross_entropy(head(Xtr[idx]), Ytr[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        sch.step()
    head.eval()
    with torch.no_grad():
        return head(torch.tensor(te_x, device=device)).argmax(1).cpu().numpy()

def visualize(ds, classes, show_images=True, tsne_n=3000):
    try:
        tr_x, tr_y = _load(ds, 'train'); te_x, te_y = _load(ds, 'test')
    except FileNotFoundError:
        print(f'[{ds}] feature 없음 — 스킵'); return
    print(f'\n===== {ds.upper()} [{MODE}] =====  train {tr_x.shape}, test {te_x.shape}')
    pred = _train_lp(tr_x, tr_y, te_x)
    acc = (pred==te_y).mean()*100
    print(f'재현 Top-1: {acc:.2f}%')

    # 1) t-SNE
    n = min(tsne_n, te_x.shape[0])
    sel = np.random.RandomState(42).choice(te_x.shape[0], n, replace=False)
    emb = TSNE(n_components=2, init='pca', perplexity=30, random_state=42).fit_transform(te_x[sel])
    plt.figure(figsize=(9,8))
    for c in range(10):
        m = te_y[sel]==c; plt.scatter(emb[m,0], emb[m,1], s=6, alpha=0.6, label=classes[c])
    plt.legend(markerscale=2, fontsize=9); plt.title(f'{ds} [{MODE}] t-SNE (Top-1 {acc:.1f}%)'); plt.show()

    # 2) confusion matrix
    cm = confusion_matrix(te_y, pred); cmn = cm/cm.sum(1, keepdims=True).clip(min=1)
    plt.figure(figsize=(8,7)); plt.imshow(cmn, cmap='Blues', vmin=0, vmax=1); plt.colorbar(fraction=0.046)
    plt.xticks(range(10), classes, rotation=45, ha='right'); plt.yticks(range(10), classes)
    for i in range(10):
        for j in range(10):
            if cmn[i,j]>0.01:
                plt.text(j,i,f'{cmn[i,j]:.2f}',ha='center',va='center',
                         color='white' if cmn[i,j]>0.5 else 'black', fontsize=7)
    plt.ylabel('True'); plt.xlabel('Pred'); plt.title(f'{ds} confusion (acc {acc:.1f}%)'); plt.show()

    # 3) per-class accuracy
    pc = [(pred[te_y==c]==c).mean()*100 for c in range(10)]
    plt.figure(figsize=(9,4)); bars = plt.bar(range(10), pc, color='steelblue')
    plt.xticks(range(10), classes, rotation=45, ha='right'); plt.ylim(0,100)
    plt.axhline(acc, color='red', ls='--', label=f'overall {acc:.1f}%')
    for b,v in zip(bars,pc): plt.text(b.get_x()+b.get_width()/2, v+1, f'{v:.0f}', ha='center', fontsize=8)
    plt.legend(); plt.ylabel('%'); plt.title(f'{ds} per-class accuracy'); plt.show()

    # 4) 샘플 예측 (실제 이미지)
    if show_images:
        try:
            from torchvision import datasets
            d = datasets.STL10(DATA, split='test', download=True) if ds=='stl10' \
                else datasets.CIFAR10(DATA, train=False, download=True)
            cor = np.where(pred==te_y)[0]; wr = np.where(pred!=te_y)[0]
            r = np.random.RandomState(0)
            sel2 = list(r.choice(cor, 8, replace=False)) + list(r.choice(wr, min(8,len(wr)), replace=False))
            fig, axes = plt.subplots(4,4, figsize=(11,12))
            for ax, i in zip(axes.flat, sel2):
                img,_ = d[int(i)]; ax.imshow(img); ax.axis('off')
                ok = pred[i]==te_y[i]
                ax.set_title(f'T:{classes[te_y[i]]}\nP:{classes[pred[i]]}',
                             color='green' if ok else 'red', fontsize=9)
            for ax in axes.flat[len(sel2):]: ax.axis('off')
            plt.suptitle(f'{ds} [{MODE}] 예측 (위2줄 정답 / 아래2줄 오답)'); plt.tight_layout(); plt.show()
        except Exception as e:
            print(f'샘플 이미지 스킵: {e}')

visualize('stl10', STL10_CLASSES)
visualize('cifar10', CIFAR10_CLASSES)

## 운영 팁 (A5000 24GB 기준)

- **MoCo v3 + ViT-S/8 (최종) / batch 2048**: VRAM ~17GB — `backbone.gradient_checkpoint: true`가
  activation을 줄여 24GB에 안전. 그래도 OOM이면 config에서 `batch_size: 1024` + `optimizer.lr: 6.0e-4`
  (lr = 1.5e-4 × batch/256). grad ckpt는 feature 추출 시 자동 무시.
- **multi-crop(2g+4l) / batch 256** (MoCo v2 트랙): 약 18-20GB 사용. OOM이면 `--batch-size 192`
  또는 config의 `gc_mode`를 `"all"`로.
- **bf16 AMP** 활성 — fp16보다 안정, GradScaler 없음. A5000(Ampere)은 `torch.cuda.is_bf16_supported()` True.
- **장시간 학습은 Terminal + nohup** — Jupyter 커널은 브라우저가 끊겨도 살아 있지만 셀 출력이
  유실되고, 커널 재시작 한 번에 학습이 죽는다. nohup이면 로그가 파일로 영구히 남는다.
  `tmux`를 쓸 수 있으면 `tmux new -s ssl` 안에서 실행하는 것도 좋다.
- **save_every + 자동 resume**: 프로세스가 죽어도 마지막 체크포인트에서 복구. Cell 3이 `outputs/`의
  마지막 `ckpt_ep*.pth`를 자동 탐색해 `--resume`. optimizer(AdamW state)·momentum encoder·
  lr scheduler step까지 복원 (smoke_test_mocov3.py [9]에서 검증).
- **시간 예산 추적**: 첫 10 epoch 평균 시간 × total_epochs로 추정 → 72h 초과 위험 시 epoch 줄여서
  시작 (cosine은 시작 시 고정 — 도중 변경 금지).
- **코드 업데이트 흐름**: VSCode 편집 → `git push` → 서버에서 Cell 2 재실행(`git pull`).
- **GPU 모니터링**: Terminal에서 `watch -n 5 nvidia-smi` 또는 `nvidia-smi dmon`.